## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
print(tf.__version__)

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Mount (clone) the github repository into the /content folder.
!git clone https://github.com/ljwg3000/UNT_MEEN.git

.

.

.
# Load Raw Data and Extract Acceleration signals

In [ ]:
NoOfData = 180

for i in range(NoOfData):

    temp_path1 = f'/content/UNT_MEEN/AI_tutorial/Dataset/Normal_{i+1}'
    temp_path2 = f'/content/UNT_MEEN/AI_tutorial/Dataset/Abnormal_{i+1}'

    exec(f"Normal_{i+1}   = pd.read_csv(temp_path1 , sep=',' , header=None)")
    exec(f"Abnormal_{i+1} = pd.read_csv(temp_path2 , sep=',' , header=None)")

- Generate single array that consists of every acceleration data (normal and abnormal)

In [ ]:
DataLength = len(Normal_1)

AccData_Nor = pd.DataFrame(np.zeros((NoOfData, DataLength)))
AccData_Abn = pd.DataFrame(np.zeros((NoOfData, DataLength)))

for i in range(NoOfData):
  exec(f"tempNormal   = Normal_{i+1}")
  exec(f"tempAbnormal = Abnormal_{i+1}")

  AccData_Nor.iloc[i,:] = tempNormal.iloc[:,1]
  AccData_Abn.iloc[i,:] = tempAbnormal.iloc[:,1]

AccData = np.array(pd.concat([AccData_Nor, AccData_Abn], axis=0))

print(AccData.shape)

# Convert Acceleration Data into Spectrogram by STFT

In [ ]:
from scipy import signal

Fs = 12800  # Sampling Frequency
f,t,AccSTFT = signal.spectrogram(AccData, Fs, nperseg = 78, noverlap = 10)

print(AccSTFT.shape)

In [ ]:
NormalSet   = AccSTFT[:NoOfData]
AbnormalSet = AccSTFT[NoOfData:]

NoOfSensor  = 1
NormalSet   = NormalSet.reshape(NormalSet.shape[0], NormalSet.shape[1], NormalSet.shape[2], NoOfSensor)
AbnormalSet = AbnormalSet.reshape(AbnormalSet.shape[0], AbnormalSet.shape[1], AbnormalSet.shape[2], NoOfSensor)

NormalSet.shape, AbnormalSet.shape

.

.

.

.

## Split Training & Test Data

In [ ]:
from sklearn.model_selection    import train_test_split

# Designate test data ratio
TestData_Ratio = 0.2

TrainData_Nor, TestData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

## Data Labling (One-hot Encoding)

- `[1,0]`: Normal
- `[1,0]`: Abnormal

In [ ]:
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0],2))
TrainLabel_Abn = np.zeros((TrainData_Abn.shape[0],2))
TestLabel_Nor  = np.zeros((TestData_Nor.shape[0],2))
TestLabel_Abn  = np.zeros((TestData_Abn.shape[0],2))

TrainLabel_Nor[:,0] = 1  # [1,0]: Normal
TrainLabel_Abn[:,1] = 1  # [0,1]: Abnormal
TestLabel_Nor[:,0]  = 1  # [1,0]: Normal
TestLabel_Abn[:,1]  = 1  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

## Data and Label Preparation

In [ ]:
TrainData  = np.concatenate([TrainData_Nor , TrainData_Abn ], axis=0)
TestData   = np.concatenate([TestData_Nor  , TestData_Abn  ], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel  = np.concatenate([TestLabel_Nor , TestLabel_Abn ], axis=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

.

.

.

.

.

# Grid search for CNN hyperparameters


### Prepare lists of hyperparameters for grid search

In [ ]:
# Hyperparameters for grid search
param_FiltS = [3, 5] # filter(kernel) size (only convolution layer)
param_FiltN = [2, 4] # number of filters   (only convolution layer)
param_Strid = [1, 2] # stride              (only convolution layer)

# Fixed hyperparameters
noOfNeuron    = 10
learningRate  = 0.0001
Epoch         = 1000

# Calculate the number of cases
NoOfCases = len(param_FiltS) * len(param_FiltN) * len(param_Strid)
NoOfCases

In [ ]:
# Create an empty dataframe to store the accuracy results
Accuracy_df = pd.DataFrame(np.zeros(shape=(NoOfCases , 4)),
                           columns=['filter size', 'number of filters', 'stride', 'Accuracy'])
Accuracy_df

In [ ]:
# Complete this function
def CNN_model(input_data, noOfNeuron, learningRate, filterSize, numOfFilters, stride):
















    return model

<details>
<summary>Click to see Answer </summary>

```python
def CNN_model(input_data, noOfNeuron, learningRate, filterSize, numOfFilters, stride):
    keras.backend.clear_session()

    model = keras.Sequential()
    model.add(keras.layers.InputLayer(shape=(input_data.shape[1],input_data.shape[2],input_data.shape[3])))       # Input layer

    model.add(keras.layers.Conv2D(filters = numOfFilters, kernel_size=(filterSize,filterSize), strides=(stride,stride), padding='same', activation='relu'))    # Convolution layer 1
    model.add(keras.layers.MaxPooling2D(pool_size = (2,2), strides=(2,2)))                                              # Pooling layer 1
    model.add(keras.layers.Conv2D(filters = numOfFilters, kernel_size=(filterSize,filterSize), strides=(stride,stride), padding='same', activation='relu'))    # Convolution layer 2
    model.add(keras.layers.MaxPooling2D(pool_size = (2,2), strides=(2,2)))                                              # Pooling layer 2

    model.add(keras.layers.Flatten())                                                                                   # Flatten layer
    model.add(keras.layers.Dense(units = noOfNeuron, activation='relu'))                                                        # Dense layer

    model.add(keras.layers.Dense(units = 2, activation='softmax'))                                                      # Output Layer

    model.compile(optimizer= keras.optimizers.Adam(learning_rate = learningRate),
                  loss=keras.losses.CategoricalCrossentropy(),
                  metrics=['accuracy'])
    return model

.

.


### Train the CNN models with different combinations of hyperparameters and save them

In [ ]:
# Callback 1: CheckProcess
EpochForPrint = 100

class CheckProcess(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        keras.callbacks.Callback()
        if epoch%EpochForPrint == 0:
            print("{} Epochs Train Acc. : {:.2f}%  ".format(epoch, logs["accuracy"]*100))

In [ ]:
# Callback 2: Early Stopping
EalryStop = keras.callbacks.EarlyStopping(
    monitor="accuracy", patience=500, restore_best_weights=True)

In [ ]:
# Initialize a count value to store the performance of each model
cnt = 0

# Iterate through all possible combinations of filter size, filter number, and stride



















<details>
<summary>Click to see Answer </summary>

```python
# Initialize a count value to store the performance of each model
cnt = 0

# Iterate through all possible combinations of filter size, filter number, and stride
for temp_FiltS in param_FiltS:          # Select each filter size in the list
    for temp_FiltN in param_FiltN:      # Select each filter number in the list
        for temp_Strid in param_Strid:  # Select each stride value in the list

            print(f"\n[Case {cnt+1}] Filter size: [{temp_FiltS},{temp_FiltS}], Num of filters: {temp_FiltN}, Stride: {temp_Strid}")
            
            # Create, train, and validate a temporary CNN model with the current combination of hyperparameters
            temp_model = CNN_model(TrainData, noOfNeuron, learningRate, temp_FiltS, temp_FiltN, temp_Strid)
            temp_model.fit(TrainData, TrainLabel, epochs=Epoch, verbose=0, callbacks=[CheckProcess()])
            Loss, Accuracy = temp_model.evaluate(TestData,  TestLabel, verbose=0)

            # Save the temporary model to a file with a corresponding name
            temp_model_name = f'CNN_FS{temp_FiltS}_FN{temp_FiltN}_St{temp_Strid}.keras'
            temp_model.save('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_CNN/' + temp_model_name)

            # Store the performance (accuracy) of the temporary model in the dataframe
            Accuracy_df.iloc[cnt, :] = [temp_FiltS, temp_FiltN, temp_Strid, Accuracy]
            cnt += 1

### Confirm the grid search results

In [ ]:
# Confirm the result of grid search
Accuracy_df

In [ ]:
# Sort the Accuracy_df by 'Accuracy' column in descending order
Accuracy_df_sorted = Accuracy_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Output the best case
Best_FiltS = int(Accuracy_df_sorted.iloc[0, 0])
Best_FiltN = int(Accuracy_df_sorted.iloc[0, 1])
Best_Strid = int(Accuracy_df_sorted.iloc[0, 2])

print(f"[Best case]\n" +
      f"Filter size   : [{Best_FiltS},{Best_FiltS}]\n" +
      f"Num of Filters: {Best_FiltN}\n" +
      f"Strides       : {Best_Strid}\n" +
      "Accuracy: %.2f" % (Accuracy_df_sorted.iloc[0, 3]))

In [ ]:
# Calculate mean and standard deviation accuracy for each filter size
mean_accuracy_FiltS = Accuracy_df.groupby(['filter size'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_FiltS

In [ ]:
# Calculate mean and standard deviation of accuracy for each number of filter
mean_accuracy_FiltN = Accuracy_df.groupby(['number of filters'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_FiltN

In [ ]:
# Calculate mean and standard deviation of accuracy for each stride
mean_accuracy_Strid = Accuracy_df.groupby(['stride'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_Strid

## Confusion matrix of the best CNN model

- TP: The number of instances where the model correctly predicted the positive class.
- TN: The number of instances where the model correctly predicted the negative class.
- FP: The number of instances where the model falsely predicted the positive class (actual negative instances).
- FN: The number of instances where the model falsely predicted the negative class (actual positive instances).

In [ ]:
# Retrieve activation function, hidden layers, and learning rate values from the first row of 'Accuracy_df_sorted'
Best_FiltS = int(Accuracy_df_sorted.iloc[0, 0])
Best_FiltN = int(Accuracy_df_sorted.iloc[0, 1])
Best_Strid = int(Accuracy_df_sorted.iloc[0, 2])

# Load the best ANN model using the retrieved hyperparameters
best_model_name = f'CNN_FS{Best_FiltS}_FN{Best_FiltN}_St{Best_Strid}.keras'
best_model = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_CNN/' + best_model_name)

# Predict the output (Robotic spot-welding condition) for the test data
Predicted = best_model.predict(TestData)

# Convert TestLabel and Predicted into vectors to calculate the confusion matrix and evaluation metrics
TestLabel_rev = np.argmax(TestLabel, axis=1)
Predicted_rev = np.argmax(Predicted, axis=1)

# Plot the confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the Best CNN Model")
plt.show()

## Evaluation metrics of the best CNN model

1. $Accuracy$: The proportion of correctly classified instances out of the total instances. It measures the overall performance of a classification model.

    - $Accuracy: (TP + TN) / (TP + TN + FP + FN)$

2. $Precision$: The proportion of true positive instances among the instances predicted as positive. It measures how well the model correctly identifies positive instances.

    - $Precision: TP / (TP + FP)$

3. $Recall$: The proportion of true positive instances among the actual positive instances. It measures the ability of the model to find all the positive instances.

    - $Recall: TP / (TP + FN)$

4. $F1 Score$: The harmonic mean of precision and recall. It provides a single score that balances both precision and recall, which is especially useful when dealing with imbalanced datasets.

    - $F1 Score: 2 * (Precision * Recall) / (Precision + Recall)$

In [ ]:
from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

# Print the evaluation metrics
print(f"Best CNN Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")